# Day 3 - Code breaks, block 2: generalisation - working on new data

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** the Iris flowers from block 1 are too easy to overfit. Here we use *two moons*: two classes shaped like
half moons, with a lot of noise. The network gets only **40 training points** and is checked on
**500 validation points** - perfect for overfitting.

## Setup - run these two cells first
The first cell creates the data.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

X_train, y_train = make_moons(n_samples=40, noise=0.3, random_state=0)    # 40 training points
X_val, y_val = make_moons(n_samples=500, noise=0.3, random_state=1)       # 500 validation points
X_train, y_train = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train)
X_val, y_val = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val)

print("training points:", len(X_train), "  validation points:", len(X_val))

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `make_model()` builds a big network (2 inputs → 128 → 128 neurons → 2 classes)
- `train(...)` trains it and returns the training loss and the validation loss of every epoch
- `plot_results(...)` draws the loss curves and what the network learned
- the other helpers are only for single code breaks

In [ ]:
import copy

from torchvision import datasets
from torchvision.transforms import v2

loss_fn = nn.CrossEntropyLoss()


def make_model(width=128, dropout=0.0):
    torch.manual_seed(0)  # same start weights for everybody
    return nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(width, 2))


def train_one_epoch(model, opt):
    model.train()                                   # train mode: dropout ON
    loss = loss_fn(model(X_train), y_train)
    opt.zero_grad()
    loss.backward()
    opt.step()
    model.eval()                                    # eval mode: dropout OFF
    with torch.no_grad():
        return loss_fn(model(X_train), y_train).item(), loss_fn(model(X_val), y_val).item()


def train(model, opt, epochs=1000):
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        train_loss, val_loss = train_one_epoch(model, opt)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
    return train_losses, val_losses


def plot_results(model, train_losses, val_losses):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(train_losses, label="training loss")
    ax1.plot(val_losses, label="validation loss")
    ax1.set_xlabel("epoch"); ax1.set_ylim(0, 4); ax1.legend()
    xx, yy = torch.meshgrid(torch.linspace(-2, 3, 200), torch.linspace(-1.5, 2, 200), indexing="xy")
    grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    model.eval()
    with torch.no_grad():
        p = model(grid).softmax(1)[:, 1].reshape(xx.shape)
    ax2.contourf(xx, yy, p, levels=20, cmap="RdBu", alpha=0.6)
    ax2.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdBu", edgecolors="k")
    ax2.set_title("what the network learned"); ax2.set_xticks([]); ax2.set_yticks([])
    plt.show()


def rounded(t):
    return [round(v, 2) + 0.0 for v in t.detach().flatten().tolist()]


saved = {}


def save_weights(model):
    saved["weights"] = copy.deepcopy(model.state_dict())


def load_best_weights(model):
    model.load_state_dict(saved["weights"])


def show_augmented(image, transform):
    fig, axes = plt.subplots(1, 7, figsize=(12, 2.2))
    axes[0].imshow(image, cmap="gray"); axes[0].set_title("original")
    for ax in axes[1:]:
        ax.imshow(transform(image)[0], cmap="gray"); ax.set_title("augmented")
    for ax in axes:
        ax.axis("off")
    plt.show()


print("helpers ready")

## 💻 Code break 1 - See overfitting
A big network, only 40 noisy points, 1000 epochs. We watch the training loss *and* the validation loss.

In [ ]:
EPOCHS = 1000      # 👈
WIDTH = 128        # 👈 neurons per hidden layer

model = make_model(width=WIDTH)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
train_losses, val_losses = train(model, opt, epochs=EPOCHS)

plot_results(model, train_losses, val_losses)
print("training loss:", round(train_losses[-1], 2), "  validation loss:", round(val_losses[-1], 2))
print("best validation loss:", round(min(val_losses), 2))

🔧 **Change it:**
1. `EPOCHS = 100` - is the gap between the two curves smaller?
2. `WIDTH = 4` - a small network. Underfitting or overfitting now?

<details><summary>👀 What you should see</summary>

- training loss goes down to **0.0**, validation loss first goes down (best **0.33**) and then *up* again - to
  **3.73** after 1000 epochs
- the boundary draws little islands around single noisy points: **overfitting** - the 40 points are learned by heart
- 100 epochs: validation loss already **1.91** - overfitting starts very early here
- width 4: training loss **0.31**, validation loss **0.76** - too small to learn every point, so much less overfitting
</details>

## 💻 Code break 2 - Weight decay: keep the weights small
The same network and the same 1000 epochs - but with `AdamW` and weight decay (knob ③ from block 1).

In [ ]:
WEIGHT_DECAY = 1.0     # 👈

model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=WEIGHT_DECAY)
train_losses, val_losses = train(model, opt)

plot_results(model, train_losses, val_losses)
print("training loss:", round(train_losses[-1], 2), "  validation loss:", round(val_losses[-1], 2))

🔧 **Change it:**
1. `WEIGHT_DECAY = 10.0` - smoother boundary?
2. `WEIGHT_DECAY = 30.0` - what happens to the *training* loss?

<details><summary>👀 What you should see</summary>

- weight decay 1: validation loss **0.62** (without weight decay it was 3.73)
- weight decay 10: **0.35** - training and validation loss almost the same, a smooth boundary
- weight decay 30: training loss rises to **0.56** - the weights are pushed too hard: **underfitting**
- on real data 0.01 is the usual start; here we need much more (1000 epochs on only 40 points)
</details>

## 💻 Code break 3 - Dropout and train mode vs eval mode
A network with dropout. We feed the **same point twice** - first in train mode, then in eval mode.

In [ ]:
DROPOUT = 0.5      # 👈 share of neurons switched off

model = make_model(dropout=DROPOUT)
point = torch.tensor([[0.5, 0.25]])          # one point

model.train()                                # train mode: dropout ON
print("train mode:", rounded(model(point)), rounded(model(point)))
model.eval()                                 # eval mode: dropout OFF
print("eval mode: ", rounded(model(point)), rounded(model(point)))

opt = torch.optim.Adam(model.parameters(), lr=0.01)
train_losses, val_losses = train(model, opt)  # train() switches train / eval mode for us
print("after training:  validation loss", round(val_losses[-1], 2))

🔧 **Change it:**
1. Run the cell again. Do the "train mode" outputs change? The "eval mode" outputs?
2. `DROPOUT = 0.8` - validation loss better or worse than without dropout (3.73)?

<details><summary>👀 What you should see</summary>

- train mode: the same point gives two **different** outputs - random neurons are switched off
- eval mode: both outputs are **identical**
- **if you forget `model.eval()`, your validation results are random**
- dropout 0.5: validation loss **2.33**; dropout 0.8: **1.24** - better than 3.73, but not enough alone. Next: early stopping.
</details>

## 💻 Code break 4 - Early stopping
After every epoch we look at the validation loss. If it is the best so far, we save the weights. If it did not
improve for `PATIENCE` epochs, we stop - and go back to the best weights.

✏️ **Fill in one line** (marked `YOUR CODE HERE`): remember the new best loss.

In [ ]:
PATIENCE = 20      # 👈

model = make_model()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
best_loss = 1000.0
epochs_without_improvement = 0

for epoch in range(1000):
    train_loss, val_loss = train_one_epoch(model, opt)
    if val_loss < best_loss:                 # a new best epoch!
        # ✏️ remember the new best loss: best_loss = ...
        pass  # YOUR CODE HERE
        best_epoch = epoch
        save_weights(model)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement = epochs_without_improvement + 1
    if epochs_without_improvement == PATIENCE:
        break                                # stop training

load_best_weights(model)
print("stopped at epoch", epoch, "  best epoch:", best_epoch, "  best validation loss:", round(best_loss, 2))

🔧 **Change it:**
1. `PATIENCE = 5`, then `PATIENCE = 200`. Does the best epoch change?

<details><summary>👀 What you should see</summary>

- stops at epoch **32**; best epoch **12** with validation loss **0.33** - much better than 3.73 after 1000 epochs
- patience 5: stops at epoch **17**, same best epoch
- patience 200: runs until epoch **212** - and still finds best epoch **12**
- if your line is missing, `best_loss` stays 1000, every epoch looks "best" and the training never stops early
</details>

## 💻 Code break 5 - Data augmentation: one image, many versions
One FashionMNIST training image and six random versions - the same transforms as on the slide.
(The first run downloads FashionMNIST, about 30 MB.)

In [ ]:
IMAGE_NR = 0       # 👈 which training image
ROTATION = 15      # 👈 largest rotation in degrees

train_tf = v2.Compose([v2.ToImage(),
    v2.RandomCrop(28, padding=3),        # shift
    v2.RandomHorizontalFlip(),           # mirror
    v2.RandomRotation(ROTATION),         # rotate
    v2.ColorJitter(brightness=0.4),      # darker / brighter
    v2.ToDtype(torch.float32, scale=True)])

fashion = datasets.FashionMNIST("data", train=True, download=True)
image, label = fashion[IMAGE_NR]
show_augmented(image, train_tf)
print("label of all 7 images:", fashion.classes[label])

🔧 **Change it:**
1. Run the cell again - new random versions every time (like in every epoch).
2. `IMAGE_NR = 1` - another image.
3. `ROTATION = 180` - would an upside-down boot be a good training image?

<details><summary>👀 What you should see</summary>

- image 0 is an **ankle boot** (image 1 is a T-shirt); shifted, mirrored, rotated, darker - but still an ankle boot
- rotation 180: some boots stand on their heads - such images never appear in the test data, so this does not help
</details>

## 💻 Code break 6 - BatchNorm with the numbers from the slide
One feature (e.g. alpha band power) in a batch of 4 EEG windows: 120, 80, 100, 100.

In [ ]:
bn = nn.BatchNorm1d(1)                                    # BatchNorm for 1 feature
batch = torch.tensor([[120.0], [80.0], [100.0], [100.0]]) # 👈 4 windows

bn.train()                                                # train mode: mean and spread of THIS batch
print("train mode:", rounded(bn(batch)))

for i in range(100):                                      # 100 more batches: BatchNorm stores averages
    bn(batch)
print("stored mean:", round(bn.running_mean.item(), 1))

bn.eval()                                                 # eval mode: uses the stored averages
print("eval mode, one window with 120:", rounded(bn(torch.tensor([[120.0]]))))

🔧 **Change it:**
1. Make the batch 10× larger: `[[1200.0], [800.0], [1000.0], [1000.0]]`. What comes out in train mode?
2. Delete the line `bn.eval()`. What happens?

<details><summary>👀 What you should see</summary>

- train mode: exactly the slide numbers **1.41, -1.41, 0, 0**; stored mean **100.0**
- eval mode: the single window 120 becomes **1.22** (the stored spread is a bit larger than 14)
- 10× larger batch: the **same** numbers - BatchNorm removes the scale
- without `bn.eval()`: an **error** - one window alone has no spread. One more reason why `model.eval()` matters.
</details>